# പാഠം 08 - ബഹുയജമാന ഡിസൈൻ പാറ്റേൺ


## സജ്ജീകരണം


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## എന്തുകൊണ്ട് മൾട്ടി-ഏജന്റ് സിസ്റ്റങ്ങൾ?

യാത്രാ പദ്ധതി പോലുള്ള യാഥാർത്ഥ്യത്തിലെ ജോലികൾ പലത്യവിധമായ വിദഗ്ധതകൾ ഉൾക്കൊള്ളുന്നു — ലജിസ്റ്റിക്‌സ്, പ്രാദേശിക അറിവ്, ബജറ്റിംഗ്, എന്നിവ. എല്ലാം കൈകാര്യം ചെയ്യാൻ ശ്രമിക്കുന്ന ഒറ്റ ഏജന്റ് വേഗത്തിൽ കൈകാര്യമാകാത്തതാവും.

മൾട്ടി-ഏജന്റ് സിസ്റ്റങ്ങൾ ഇത് **വിശേഷിക്‌ഷണത്തിലൂടെ** പരിഹരിക്കുന്നു: ഓരോ ഏജന്റും ഒരു വിദഗ്ധതാ മേഖലയിലേക്ക് കേന്ദ്രീകരിക്കുകയും, ഒരു ജനറലിസ്റ്റിനേക്കാൾ ഉയർന്ന നിലവാരമുള്ള ഫലങ്ങൾ നൽകുകയും ചെയ്യുന്നു. അവ **വ്യാപ്തിയുടെയും** മെച്ചപ്പെടുത്തുകയുമാണ് — പുതിയ ഏജന്റുകൾ (ഉദാഹരണത്തിന്, ഫ്ലൈറ്റ് വിദഗ്ധൻ, റെസ്റ്റോറന്റ് നിരൂപകൻ) എന്നിങ്ങനെ നിലവിലുള്ള വർക്ക്‌ഫ്ലോ പുനഃരചയിക്കാതെ ചേർക്കാവുന്നതാണ്. ഏജന്റുകൾ ഘടനാപരമായ പൈപ്പ്ലൈനിലൂടെ ചേർന്ന് പ്രവർത്തിക്കുന്നു, ഒരു ഏജന്റിൽ നിന്നും അടുത്തയിലേക്ക് സാന്ദർഭ്യം കൈമാറുന്നു.


## പ്രത്യേക ഏജന്റുകൾ സൃഷ്ടിക്കൽ


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Building a Sequential Workflow

`WorkflowBuilder` നിങ്ങളെ ഏജന്റുകളെ ഒരു ദിശാനിർദ്ദേശ ഗ്രാഫിൽ വയർ ചെയ്യാൻ അനുവദിക്കുന്നു. ഇവിടെ ഞങ്ങൾ ഒരു ലളിതമായ രണ്ട് ഘട്ട_pipeline_ സൃഷ്ടിക്കുന്നു: **TravelPlanner** യാത്രാ പദ്ധതി രൂപകല്‌പന ചെയ്യുന്നു, പിന്നീട് **TravelConcierge** അതിനെ അവലോകനം ചെയ്ത് മെച്ചപ്പെടുത്തുന്നു.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## വർക്ക്‌ഫ്ലോയിലേക്ക് കൂടുതൽ ഏജന്റുമാർ ചേർക്കുക

മൾട്ടി- ഏജന്റ് പാറ്റേണിൻ്റെ ഏറ്റവും വലിയ നേട്ടം അതിനെ എളുപ്പത്തിൽ വികസ്വരമാക്കാൻ കഴിവുള്ളതാണെന്ന് പറയാം. താഴെ ഒരു **BudgetReviewer** ഏജന്റ് ചേർക്കുന്നു, ഇതു യാത്രികന്റെ ബജറ്റിന് വിരുദ്ധമായി യോജിപ്പിച്ച പദ്ധതിയെ പരിശോദിക്കുന്നു, ചില വസ്തുക്കൾ ചെലവ് പരിധി മുറിച്ചുപോയാൽ അടയാളപ്പെടുത്തുന്നു, ഫലപ്രദമായ പണം ലാഭിക്കാനുള്ള മറ്റ് മാർഗങ്ങൾ നിർദ്ദേശിക്കുന്നു. ഇപ്പോൾ വർക്ക്‌ഫ്ലോ പരമ്പരാഗതമായി മൂന്ന് ഏജന്റുമാർ പ്രവർത്തിക്കുന്നു:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## സംക്ഷേപം

ഈ പാഠത്തിൽ നിങ്ങൾ പഠിച്ചത്:

1. **പ്രത്യേക ഏജന്റുകൾ സൃഷ്‌ടിക്കുക** — ഓരോന്നും ഒരു കേന്ദ്രീകൃത റോളിനായി (പ്ലാനിംഗ്, കോൺസിയർജ്, ബഡ്ജറ്റ് അവലോകനം).
2. **`WorkflowBuilder`വും `add_edge`ഉം ഉപയോഗിച്ച് ഏജന്റുകൾ ഒരു ക്രമാതീത പ്രവൃത്തിയിലേക്ക് ബന്ധിപ്പിക്കുക**.
3. **ഒരു ബഹുഏജന്റ് പൈപ്പ്ലൈൻ ഔട്ട്‌പുട്ട് സ്റ്റ്രീം ചെയ്യുക**, ഏജന്റ് ആരാണെന്ന് ട്രാക്ക് ചെയ്യുക.
4. **ഒരു പ്രവൃത്തി ഫെൽക്സ് വിപുലീകരിക്കുക** പഴയ ഏജന്റുകൾ മാറ്റാതെ പുതിയ ഏജന്റുകൾ ചേർത്ത്.

ബഹുഏജന്റ് രൂപകൽപ്പന മാതൃക ഓരോ ഏജന്റിനെയും ലളിതമാക്കുന്നുവെങ്കിലും, വലിയ, കൂടുതൽ സൂക്ഷ്മമായി പരിശോധിക്കപ്പെട്ട ഫലങ്ങൾ ഒരു ഏജന്റു മാത്രം നേടാനാകാത്ത വിധത്തിൽ ഉത്പാദിപ്പിക്കുന്നു.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ഡിസ്ക്ലെയ്മർ**:  
ഈ ഡോക്യുമെന്റ് AI വിവർത്തന സേവനമായ [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് വിവർത്തനം ചെയ്തതാണ്. നാം ശരിയായി വിവർത്തനം ചെയ്യാൻ പരിശ്രമിക്കുമ്പോഴും, സ്വയമേവയുടെ വിവർത്തനത്തിൽ വീഴ്ചകൾ അല്ലെങ്കിൽ തെറ്റുകൾ ഉണ്ടാകാവുന്ന സാഹചര്യത്തെ ശ്രദ്ധിക്കുക. അതിന്റെ റൂട്ട് ഭാഷയിൽ ഉള്ള ഒറിജിനൽ ഡോക്യുമെന്റ് തന്നെയാണ് ഏറ്റവും വിശ്വാസയോഗ്യമായ ഉറവിടം. അത്യവശ്യ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മാനുഷിക വിവർത്തന സേവനം ഉപയോജിപ്പിക്കുന്നത് സുരക്ഷിതമാണ്. ഈ വിവർത്തനം ഉപയോഗിക്കുന്നതിൽനിന്നും ഉണ്ടാകാവുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ വ്യാഖ്യാന ദോഷങ്ങൾക്ക് ഞങ്ങൾ ഉത്തരവാദിത്വം സ്വീകരിക്കുന്നില്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
